# K-Means GPU Benchmark: CPU vs GPU
**Ordinea recomandata:** rulati celulele in ordine de sus in jos.

Asigurati-va ca ati selectat: `Runtime > Change runtime type > GPU (T4)`

## 1. Setup — Clone repo si instalare dependinte

In [ ]:
# Cloneaza repo-ul
!git clone https://github.com/andreisugu/KMeans-GPU-Benchmark.git
%cd KMeans-GPU-Benchmark

In [ ]:
# Instalare dependinte Python (sklearn, pandas, matplotlib)
!pip install -r requirements.txt -q

# Instalare RAPIDS cuML (necesita GPU runtime)
!pip install cuml-cu11 --extra-index-url=https://pypi.nvidia.com -q
print('✅ Dependinte instalate')

In [ ]:
# Verifica GPU disponibil
!nvidia-smi
import torch; print('CUDA disponibil:', torch.cuda.is_available())

## 2. Benchmark CPU — scikit-learn (Python)

In [ ]:
%cd src/baseline
!python benchmark_cpu.py --skip-large --output ../../results/results_cpu.csv
%cd ../..

## 3. Benchmark CPU — C++ Secvential (compilare + rulare)

In [ ]:
%cd src/baseline
!make clean && make
!./kmeans_seq --n 10000 --d 2 --k 5 --output ../../results/results_cpp.csv
!./kmeans_seq --n 100000 --d 16 --k 10 --output ../../results/results_cpp.csv
%cd ../..

## 4. Benchmark GPU — RAPIDS cuML

In [ ]:
%cd src/rapids
!python benchmark_rapids.py --skip-large --output ../../results/results_rapids.csv
%cd ../..

## 5. Vizualizare rezultate comparative

In [ ]:
%cd results
!python plot_results.py
%cd ..

from IPython.display import Image, display
display(Image('results/time_comparison.png'))
display(Image('results/speedup_chart.png'))

## 6. Validare corectitudine (inertia CPU vs GPU)
Verifica ca implementarile converg la solutii similare.

In [ ]:
import pandas as pd
import os

dfs = []
for f in ['results/results_cpu.csv', 'results/results_cpp.csv', 'results/results_rapids.csv']:
    if os.path.exists(f):
        dfs.append(pd.read_csv(f))

df = pd.concat(dfs)
print(df[['Platform','N_Samples','D_Features','K_Clusters','Time_Seconds','Iterations','Inertia']].to_string(index=False))